In [139]:
import pandas as pd
import numpy  as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import make_blobs

import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [140]:
x, y = make_blobs(
    n_samples=10000,      # عدد العينات
    centers=2,            # عدد المجموعات (افتراضي=3 إذا لم تُحدد)
    n_features=2,         # عدد الخصائص (الأعمدة)
    random_state=42       # لجعل النتائج قابلة لإعادة الإنتاج
)

In [141]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [142]:
x_train = np.array(x_train)
x_test = np.array(x_test)
y_train = np.array(y_train)
y_test = np.array(y_test)

In [143]:
import numpy as np
from sklearn.metrics import accuracy_score

# ==========================
# Initialisation des poids pour 3 couches (2 cachées + sortie)
# ==========================
def initialisation_MLP(input_dim, hidden1_dim, hidden2_dim, output_dim):
    np.random.seed(6)
    w1 = np.random.rand(input_dim, hidden1_dim)
    b1 = np.zeros((1, hidden1_dim))
    w2 = np.random.rand(hidden1_dim, hidden2_dim)
    b2 = np.zeros((1, hidden2_dim))
    w3 = np.random.rand(hidden2_dim, output_dim)
    b3 = np.zeros((1, output_dim))
    return w1,b1,w2,b2,w3,b3

# ==========================
# Fonctions d'activation
# ==========================
def sigmoid(Z):
    return 1/(1+np.exp(-Z))

def sigmoid_derivative(A):
    return A*(1-A)

def ReLU(Z):
    return np.maximum(0,Z)

def ReLU_derivative(Z):
    return (Z>0).astype(float)

# ==========================
# Forward propagation
# ==========================
def forward_propagation(X, w1,b1,w2,b2,w3,b3):
    Z1 = np.dot(X, w1) + b1
    A1 = ReLU(Z1)
    Z2 = np.dot(A1, w2) + b2
    A2 = ReLU(Z2)
    Z3 = np.dot(A2, w3) + b3
    A3 = sigmoid(Z3)  # Sortie binaire
    cache = (Z1,A1,Z2,A2,Z3,A3)
    return A3, cache

# ==========================
# Calcul de la perte
# ==========================
def Loss(Y, A):
    Y = Y.reshape(-1,1)  # Assurer forme (n_samples,1)
    A = A.reshape(-1,1)
    n = Y.shape[0]
    epsilon = 1e-10
    L = (-1/n) * np.sum(Y*np.log(A+epsilon) + (1-Y)*np.log(1-A+epsilon))
    return L

# ==========================
# Backpropagation
# ==========================
def backpropagation(X,Y,cache,w1,b1,w2,b2,w3,b3,learning_rate):
    Z1,A1,Z2,A2,Z3,A3 = cache
    n = X.shape[0]
    
    Y = Y.reshape(-1,1)   # s'assurer que Y est (n_samples,1)
    A3 = A3.reshape(-1,1)
    
    # Gradient sortie
    dZ3 = A3 - Y
    dw3 = (1/n) * np.dot(A2.T, dZ3)
    db3 = (1/n) * np.sum(dZ3, axis=0, keepdims=True)
    
    # Gradient couche cachée 2
    dA2 = np.dot(dZ3, w3.T)
    dZ2 = dA2 * ReLU_derivative(Z2)
    dw2 = (1/n) * np.dot(A1.T, dZ2)
    db2 = (1/n) * np.sum(dZ2, axis=0, keepdims=True)
    
    # Gradient couche cachée 1
    dA1 = np.dot(dZ2, w2.T)
    dZ1 = dA1 * ReLU_derivative(Z1)
    dw1 = (1/n) * np.dot(X.T, dZ1)
    db1 = (1/n) * np.sum(dZ1, axis=0, keepdims=True)
    
    # Mise à jour des poids et biais
    w1 -= learning_rate * dw1
    b1 -= learning_rate * db1
    w2 -= learning_rate * dw2
    b2 -= learning_rate * db2
    w3 -= learning_rate * dw3
    b3 -= learning_rate * db3
    
    return w1,b1,w2,b2,w3,b3

# ==========================
# Perceptron multicouche
# ==========================
def Perceptron_Multicouche(X_train,Y_train,X_test,Y_test,learning_rate,epoch,
                           hidden1_dim=8, hidden2_dim=5):
    input_dim = X_train.shape[1]
    output_dim = 1
    
    # Initialisation des poids et biais
    w1,b1,w2,b2,w3,b3 = initialisation_MLP(input_dim, hidden1_dim, hidden2_dim, output_dim)
    loss_history = []
    
    for i in range(epoch):
        # Forward propagation
        A3, cache = forward_propagation(X_train, w1,b1,w2,b2,w3,b3)
        L = Loss(Y_train, A3)
        loss_history.append(L)
        
        # Backpropagation avec tous les biais transmis
        w1,b1,w2,b2,w3,b3 = backpropagation(X_train,Y_train,cache,
                                             w1,b1,w2,b2,w3,b3,
                                             learning_rate)
    
    # Evaluation sur le jeu de test
    y_pred,_ = forward_propagation(X_test,w1,b1,w2,b2,w3,b3)
    y_pred = np.where(y_pred>0.6,1,0)
    accuracy = accuracy_score(Y_test,y_pred)
    
    return np.mean(loss_history), accuracy


In [144]:
learning_rate = 0.01
epoch = 1000

In [145]:
loss,acc =Perceptron_Multicouche(X_train=x_train,
                                 Y_train=y_train,
                                    X_test=x_test,
                                    Y_test=y_test,
                                    learning_rate=learning_rate,
                                    epoch=epoch,
                                    hidden1_dim=8,
                                    hidden2_dim=8)

In [146]:

print(f"Loss : {loss:.2f}")
print(f'Accuracy: {acc:.2f}')

Loss : 0.06
Accuracy: 1.00
